# GPT Training Notebook

Clean training notebook for your custom GPT architecture.

In [2]:
import os
import sys
import torch
import tiktoken
import torch.nn.functional as F
sys.path.append(os.path.abspath(".."))
from compact_gpt_architecture import (
    GPTModel,
    create_dataloader,
    generate_text_simple
)


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

tokenizer = tiktoken.get_encoding("o200k_base")


Device: cuda


In [4]:
GPT_CONFIG_124M = {
    "vocab_size": tokenizer.n_vocab,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}


## Load Dataset

In [5]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

split_idx = int(0.9 * len(text_data))

train_text = text_data[:split_idx]
val_text = text_data[split_idx:]

print("Train characters:", len(train_text))
print("Validation characters:", len(val_text))


Train characters: 164289
Validation characters: 18255


## Create Dataloaders

In [6]:
train_loader = create_dataloader(
    train_text,
    batch_size=8,
    max_length=256,
    stride=128,
    shuffle=True,
    drop_last=True,
    num_workers=0
)

val_loader = create_dataloader(
    val_text,
    batch_size=8,
    max_length=256,
    stride=128,
    shuffle=False,
    drop_last=False,
    num_workers=0
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))


Train batches: 66
Validation batches: 7


## Initialize Model

In [7]:
torch.manual_seed(123)

model = GPTModel(GPT_CONFIG_124M)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())

print(f"Total Parameters: {total_params:,}")


Total Parameters: 392,454,144


## Optimizer + Scheduler

In [8]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.95),
    eps=1e-8,
    weight_decay=0.1
)

num_epochs = 25

total_training_steps = len(train_loader) * num_epochs

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=total_training_steps
)

scaler = torch.cuda.amp.GradScaler()


/tmp/ipykernel_887/2804228104.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


## Training Loop

In [9]:
for epoch in range(num_epochs):

    model.train()

    total_loss = 0

    for input_batch, target_batch in train_loader:

        input_batch = input_batch.to(device)
        target_batch = target_batch.to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast():

            logits = model(input_batch)

            loss = F.cross_entropy(
                logits.flatten(0, 1),
                target_batch.flatten()
            )

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        scaler.step(optimizer)
        scaler.update()

        scheduler.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {avg_train_loss:.4f}")

    model.eval()

    val_loss = 0

    with torch.no_grad():

        for input_batch, target_batch in val_loader:

            input_batch = input_batch.to(device)
            target_batch = target_batch.to(device)

            logits = model(input_batch)

            loss = F.cross_entropy(
                logits.flatten(0, 1),
                target_batch.flatten()
            )

            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(f"Validation Loss: {avg_val_loss:.4f}")
    print("-" * 50)


/tmp/ipykernel_887/1546750054.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1
Train Loss: 6.7303
Validation Loss: 7.0142
--------------------------------------------------
Epoch 2
Train Loss: 4.7239
Validation Loss: 6.5667
--------------------------------------------------
Epoch 3
Train Loss: 3.9238
Validation Loss: 6.3904
--------------------------------------------------
Epoch 4
Train Loss: 3.3516
Validation Loss: 6.3934
--------------------------------------------------
Epoch 5
Train Loss: 2.8325
Validation Loss: 6.4085
--------------------------------------------------
Epoch 6
Train Loss: 2.3056
Validation Loss: 6.5551
--------------------------------------------------
Epoch 7
Train Loss: 1.7615
Validation Loss: 6.7502
--------------------------------------------------
Epoch 8
Train Loss: 1.2530
Validation Loss: 7.0065
--------------------------------------------------
Epoch 9
Train Loss: 0.8325
Validation Loss: 7.3159
--------------------------------------------------
Epoch 10
Train Loss: 0.5295
Validation Loss: 7.5971
------------------------------

## Save Checkpoint

In [10]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "config": GPT_CONFIG_124M,
}, "gpt_checkpoint.pth")

print("Checkpoint saved.")


Checkpoint saved.


## Text Generation

In [15]:
model.eval()

start_context = "সিলেট ওসমানী মেডিকেল কলেজ ছাত্রদল নেতা তাওহীদুল"

encoded = tokenizer.encode(
    start_context,
    allowed_special={"<|endoftext|>", "<|endofprompt|>"}
)

encoded_tensor = torch.tensor(
    encoded,
    dtype=torch.long
).unsqueeze(0).to(device)

out = generate_text_simple(
    model=model,
    idx=encoded_tensor,
    max_new_tokens=50,
    context_size=GPT_CONFIG_124M["context_length"]
)

decoded_text = tokenizer.decode(
    out.squeeze(0).tolist()
)

print(decoded_text)


সিলেট ওসমানী মেডিকেল কলেজ ছাত্রদল নেতা তাওহীদুল ইসলাম বলেন, এই রেলপথ নির্মাণে রাজনৈতিক একটা প্রভাব পড়বে। নির্বাচনী যে অঙ্গীকার করেছিলাম সেটা আমরা পূরণ করেছি। এতে উন্নয়নের যে ধারা তার সঙ্গে মানুষ থাকবে বলে আশা করি।
এ
